# Interactive Land Value Maps by Neighborhood

In [13]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import folium
import plotly.graph_objects as go
from pyproj import Transformer
import pyreadr
import os

In [14]:
# Load data
result = pyreadr.read_r('/Volumes/ssd_externo/UEL MESTRADO 2020/Dissertação/Simulações em R/GAMLSS/pred_geral')
df = result[None]
print(f"Total points: {len(df)}")
print(f"Columns: {list(df.columns)[:10]}...")

Total points: 89089
Columns: ['x', 'y', 'jan.2000', 'jul.2000', 'jan.2001', 'jul.2001', 'jan.2002', 'jul.2002', 'jan.2003', 'jul.2003']...


In [15]:
# Load neighborhoods with proper encoding
shp_path = '/Users/fjcosta/Documents/landCoverlandValue/bairros/BairrosLondrina.shp'
neighborhoods = gpd.read_file(shp_path, encoding='utf-8')
neighborhoods_29192 = neighborhoods.to_crs(epsg=29192)
print(f"Neighborhoods loaded: {len(neighborhoods_29192)}")
print(f"Sample neighborhood names: {neighborhoods_29192['BAIRRO'].head(10).tolist()}")

Neighborhoods loaded: 66
Sample neighborhood names: ['Alpes', 'Coliseu', 'Shangri-lá', 'Vila Nova', 'Vila Recreio', 'Vila Casoni', 'Centro', 'Vila Brasil', 'Ipiranga', 'Nova Esperança']


In [16]:
# Create points GeoDataFrame
geometry = [Point(xy) for xy in zip(df['x'], df['y'])]
gdf_points = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:29192')
print(f"GeoDataFrame created with {len(gdf_points)} points")

GeoDataFrame created with 89089 points


In [17]:
# Spatial join
points_with_neighborhood = gpd.sjoin(
    gdf_points, 
    neighborhoods_29192[['BAIRRO', 'geometry']], 
    how='left', 
    predicate='within'
)
print(f"Points with neighborhood: {points_with_neighborhood['BAIRRO'].notna().sum()}")
print(f"Points without neighborhood: {points_with_neighborhood['BAIRRO'].isna().sum()}")

Points with neighborhood: 79198
Points without neighborhood: 9892


In [18]:
# Get temporal columns
temporal_cols = [col for col in df.columns if col.startswith(('jan.', 'jul.'))]
print(f"Temporal columns: {len(temporal_cols)}")
print(f"From {temporal_cols[0]} to {temporal_cols[-1]}")

Temporal columns: 44
From jan.2000 to jul.2021


In [19]:
# Calculate neighborhood MEDIANS (not means)
neighborhood_temporal_data = {}

for bairro in neighborhoods_29192['BAIRRO'].unique():
    points_in_bairro = points_with_neighborhood[points_with_neighborhood['BAIRRO'] == bairro]
    
    if len(points_in_bairro) > 0:
        median_values = points_in_bairro[temporal_cols].median()
        neighborhood_temporal_data[bairro] = median_values.to_dict()
    else:
        neighborhood_temporal_data[bairro] = None

with_data = sum(1 for v in neighborhood_temporal_data.values() if v is not None)
without_data = sum(1 for v in neighborhood_temporal_data.values() if v is None)
print(f"With data: {with_data}")
print(f"Without data: {without_data}")

With data: 66
Without data: 0


In [20]:
# Function to create Plotly chart
def create_time_series_chart(bairro_name, temporal_data):
    if temporal_data is None:
        return "<div style='padding:20px; text-align:center; font-family:Arial;'><b>No data available</b></div>"
    
    periods = list(temporal_data.keys())
    values = list(temporal_data.values())
    
    period_labels = []
    for period in periods:
        month, year = period.split('.')
        month_abbr = 'Jan' if month == 'jan' else 'Jul'
        period_labels.append(f"{month_abbr}/{year}")
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=period_labels,
        y=values,
        mode='lines+markers',
        name='Neighborhood Median Value',
        line=dict(color='#1f77b4', width=2),
        marker=dict(size=4)
    ))
    
    fig.update_layout(
        title=dict(
            text=f"<b>{bairro_name}</b><br><span style='font-size:12px; font-weight:normal; color:#666;'>Neighborhood Median Land Value</span>",
            x=0.5,
            xanchor='center',
            font=dict(size=16)
        ),
        xaxis_title='Period',
        yaxis_title='Median Value (R$/m²)',
        hovermode='x unified',
        width=650,
        height=420,
        margin=dict(l=60, r=40, t=90, b=100),
        font=dict(size=11),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            tickangle=-45
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgray'
        )
    )
    
    chart_html = fig.to_html(
        include_plotlyjs='cdn',
        config={'displayModeBar': False},
        div_id=None
    )
    
    # Add explanatory text below chart
    info_text = """
    <div style='padding: 15px; font-family: Arial, sans-serif; font-size: 12px; 
                background-color: #f8f9fa; border-top: 2px solid #dee2e6; margin-top: 10px;'>
        <p style='margin: 0 0 8px 0; font-weight: bold; color: #495057;'>Estimation under the following paradigm:</p>
        <ul style='margin: 0; padding-left: 20px; color: #6c757d; line-height: 1.6;'>
            <li>Relief: flat</li>
            <li>Total area: 377 m²</li>
            <li>Pavement: Asphalt</li>
            <li>Type: Parcel</li>
        </ul>
    </div>
    """
    
    full_html = chart_html + info_text
    
    return full_html

In [21]:
# Generate all charts
neighborhood_charts = {}
for bairro, data in neighborhood_temporal_data.items():
    chart_html = create_time_series_chart(bairro, data)
    neighborhood_charts[bairro] = chart_html

print(f"Charts generated: {len(neighborhood_charts)}")

Charts generated: 66


In [22]:
# Transform to lat/lon and create map
neighborhoods_latlon = neighborhoods_29192.to_crs(epsg=4326)
center_lat = neighborhoods_latlon.geometry.centroid.y.mean()
center_lon = neighborhoods_latlon.geometry.centroid.x.mean()

print(f"Map center: ({center_lat:.4f}, {center_lon:.4f})")

Map center: (-23.3094, -51.1607)


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_72716/3258916224.py:3: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_72716/3258916224.py:4: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [23]:
# Create map with popups
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite'
)

# Add neighborhoods with popups
for idx, row in neighborhoods_latlon.iterrows():
    bairro_name = row['BAIRRO']
    chart_html = neighborhood_charts.get(bairro_name, "<div>Error</div>")
    
    # Create IFrame for popup
    iframe = folium.IFrame(html=chart_html, width=700, height=570)
    popup = folium.Popup(iframe, max_width=700)
    
    folium.GeoJson(
        row.geometry,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#ffffff',
            'weight': 1.5,
            'fillOpacity': 0.1
        },
        highlight_function=lambda x: {
            'fillColor': '#ffff00',
            'fillOpacity': 0.3,
            'weight': 3
        },
        tooltip=folium.Tooltip(bairro_name),
        popup=popup
    ).add_to(m)

print(f"Added {len(neighborhoods_latlon)} neighborhoods with popups")

Added 66 neighborhoods with popups


In [24]:
# Save final map
output_dir = '/Users/fjcosta/Documents/landCoverlandValue/landvalue/temporal/'
final_path = os.path.join(output_dir, 'neighborhood_temporal_analysis.html')
m.save(final_path)

file_size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\n{'='*60}")
print(f"Final map saved: {final_path}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"{'='*60}")


Final map saved: /Users/fjcosta/Documents/landCoverlandValue/landvalue/temporal/neighborhood_temporal_analysis.html
File size: 1.48 MB
